# ML Pipeline - Phase 1: Phân tích Khám phá Dữ liệu (EDA) & Tiền Xử Lý (PTB-XL)

Thực hiện theo quy trình chuẩn:
1. **Nạp dữ liệu PTB-XL**: `data/features/ptbxl_features.csv`.
2. **Rà soát thuộc tính phân loại**: One-Hot Encoding cho các biến phân loại (nếu có).
3. **Khảo sát độ biến thiên**: Thống kê độ lệch chuẩn & độ biến thiên các đặc trưng.
4. **Mean Imputation & Scaling**: Xử lý dữ liệu khuyết bằng `SimpleImputer(strategy='mean')` và chuẩn hóa biên độ.
5. **Xuất tập Scaled vào data/processed/**: Min-Max Scaling (`ptbxl_minmax_scaled.csv`) vs Z-Score (`ptbxl_zscore_scaled.csv`).

In [ ]:
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (16, 6)
print('✅ Nạp thành công thư viện EDA cho PTB-XL!')

### 1. Nạp Dữ Liệu Đặc Trưng & Kiểm Tra Cấu Trúc

In [ ]:
import os, pandas as pd
data_candidates = ['../../../data/features/ptbxl_features.csv', '../../data/features/ptbxl_features.csv', 'data/features/ptbxl_features.csv']
data_path = next((p for p in data_candidates if os.path.exists(p)), None)
if not data_path:
    raise FileNotFoundError('❌ Không tìm thấy file ptbxl_features.csv trong data/features!')
df_raw = pd.read_csv(data_path)
print(f'✅ Nạp dữ liệu thành công từ: {{data_path}}')
print(f'Kích thước dữ liệu: {{df_raw.shape[0]}} mẫu | {{df_raw.shape[1]}} cột')
cat_cols = df_raw.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    df_processed = pd.get_dummies(df_raw, columns=cat_cols, drop_first=True)
    print('✅ Đã áp dụng One-Hot Encoding cho các cột phân loại!')
else:
    df_processed = df_raw.copy()
    print('ℹ️ Tất cả đặc trưng đều là dạng số (Numerical).')

### 2. Thống Kê Độ Biến Thiên Các Đặc Trưng (Variance Analysis)

In [ ]:
feature_cols = [c for c in df_processed.columns if c != 'status']
X_raw = df_processed[feature_cols]
y = df_processed['status']
stats_df = pd.DataFrame({
    'Mean': X_raw.mean(),
    'Std_Dev': X_raw.std(),
    'Variance': X_raw.var(),
    'Min': X_raw.min(),
    'Max': X_raw.max(),
    'Range': X_raw.max() - X_raw.min()
}).sort_values(by='Variance', ascending=False)
print('--- BẢNG THỐNG KÊ ĐỘ BIẾN THIÊN PTB-XL ---\n')
print(stats_df.round(4).to_string())

### 3. Xử lý Mean Imputation & Xuất Dữ Liệu Scaled vào `data/processed/`

In [ ]:
# Imputation
imputer = SimpleImputer(strategy='mean')
X_imputed = pd.DataFrame(imputer.fit_transform(X_raw), columns=feature_cols)

# Min-Max Scaling
scaler_mm = MinMaxScaler()
X_minmax = pd.DataFrame(scaler_mm.fit_transform(X_imputed), columns=feature_cols)
df_minmax = pd.concat([X_minmax, y.reset_index(drop=True)], axis=1)

# Z-Score Scaling
scaler_z = StandardScaler()
X_zscore = pd.DataFrame(scaler_z.fit_transform(X_imputed), columns=feature_cols)
df_zscore = pd.concat([X_zscore, y.reset_index(drop=True)], axis=1)

# Lưu vào data/processed/
proc_dir_candidates = ['../../../data/processed', '../../data/processed', 'data/processed']
out_dir = next((d for d in proc_dir_candidates if os.path.exists(os.path.dirname(d))), '../../data/processed')
os.makedirs(out_dir, exist_ok=True)

mm_path = os.path.join(out_dir, 'ptbxl_minmax_scaled.csv')
z_path = os.path.join(out_dir, 'ptbxl_zscore_scaled.csv')
df_minmax.to_csv(mm_path, index=False)
df_zscore.to_csv(z_path, index=False)
print(f'🎉 [PTB-XL] Min-Max Scaled -> {mm_path}')
print(f'🎉 [PTB-XL] Z-Score Scaled -> {z_path}')

### 4. Trực Quan Hóa So Sánh Phân Phối SDNN (Thô vs Min-Max vs Z-Score)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.kdeplot(X_raw['SDNN'], ax=axes[0], color='blue', fill=True)
axes[0].set_title('PTB-XL - SDNN Gốc (Original)', fontsize=12, fontweight='bold')

sns.kdeplot(X_minmax['SDNN'], ax=axes[1], color='orange', fill=True)
axes[1].set_title('PTB-XL - Min-Max Scaled [0, 1]', fontsize=12, fontweight='bold')

sns.kdeplot(X_zscore['SDNN'], ax=axes[2], color='green', fill=True)
axes[2].set_title('PTB-XL - Z-Score Scaled (Mean=0, Std=1)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()